In [1]:
import pandas as pd
df = pd.read_csv(r"D:\Downloads\insurance_data.csv")

In [2]:
df.head()

,age,affordibility,bought_insurance
0,22,1,0
1,25,0,0
2,47,1,1
3,52,0,0
4,46,1,1


In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df[["age", "affordibility"]], df.bought_insurance, test_size=0.25)

In [6]:
len(X_train)

21

In [7]:
len(X_test)

7

**This is used for the scaling...**

In [8]:
X_train = X_train.copy()
X_train["age"] = X_train["age"]/100

X_test = X_test.copy()
X_test["age"] = X_test["age"]/100

In [9]:
X_train

,age,affordibility
10,0.18,1
18,0.19,0
20,0.21,1
0,0.22,1
5,0.56,1
13,0.29,0
3,0.52,0
17,0.58,1
12,0.27,0
4,0.46,1


In [14]:
from tensorflow import keras

model = keras.Sequential([
    keras.layers.Dense(1, input_shape=(2,), activation="sigmoid")
])

model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])

model.fit(X_train, y_train, epochs=5000)

Epoch 1/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.5714 - loss: 0.8272
Epoch 2/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.5714 - loss: 0.8268
Epoch 3/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step - accuracy: 0.5714 - loss: 0.8264
Epoch 4/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - accuracy: 0.5714 - loss: 0.8260
Epoch 5/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step - accuracy: 0.5714 - loss: 0.8256
Epoch 6/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step - accuracy: 0.5714 - loss: 0.8253
Epoch 7/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step - accuracy: 0.5714 - loss: 0.8249
Epoch 8/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.5714 - loss: 0.8245
Epoch 9/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.5714 - loss: 0.8241
Epoch 10/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - accuracy: 0.5714 - loss: 0.8237
Epoch 11/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step - accuracy: 0.5714 - loss: 0.8234
Epoch 12/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accur

In [15]:
model.evaluate(X_test, y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 275ms/step - accuracy: 1.0000 - loss: 0.4378


[0.43776533007621765, 1.0]

In [ ]:
coef, intercept = model.get_weights()
coef, intercept

#This gives the w1 and w2 also the bias
#for the neural network

(array([[3.7773602],
        [1.1495365]], dtype=float32),
 array([-2.5030148], dtype=float32))

**Building the neural network with python only**

In [29]:
import numpy as np
def sigmoid(x):
    return 1/(1+np.exp(-x))

In [30]:
sigmoid(21)

np.float64(0.9999999992417439)

In [31]:
def prediction_function(age, affordibility):
    weighted_sum = coef[0]*age + coef[1]*affordibility + intercept
    return sigmoid(weighted_sum)

In [34]:
prediction_function(.18, 1)

array([0.33770162], dtype=float32)

**Writing the gradient descent function from the scratch**

In [35]:
def log_loss(y_true, y_predicted):
    epsilon = 1e-15
    y_predicted_new = [max(i, epsilon) for i in y_predicted]
    y_predicted_new = [min(i, 1-epsilon) for i in y_predicted]
    y_predicted_new = np.array(y_predicted_new)
    return -np.mean(y_true*np.log(y_predicted_new)+(1-y_true)*np.log(1-y_predicted_new))

_From the gradient descent we can find out the desired weights for our neural network which gives best prediction and minimal loss_

In [39]:
def gradient_descent(age, affordibility, y_true, epochs, loss_threshold):
    w2=w1=1
    bias = 0
    rate = 0.5
    n = len(age)

    for i in range(epochs):
        weighted_sum = w1*age + w2*affordibility + bias
        y_predicted = sigmoid(weighted_sum)
        loss = log_loss(y_true, y_predicted)

        w1_derivative = (1/n)*(np.dot(np.transpose(age), (y_predicted-y_true)))
        w2_derivative = (1/n)*(np.dot(np.transpose(affordibility), (y_predicted-y_true)))
        bias_derivative = (1/n)*np.sum(y_predicted-y_true)

        w1 = w1-rate*w1_derivative
        w2 = w2-rate*w2_derivative
        bias = bias-rate*bias_derivative

        print(f"epochs : {i}, w1 : {w1}, w2 : {w2}, bias : {bias}, loss : {loss}")

        if loss<=loss_threshold:
            break

    return w1, w2, bias

In [40]:
gradient_descent(X_train["age"], X_train["affordibility"], y_train, 1000, 0.4960)

epochs : 0, w1 : 0.9661065669301468, w2 : 0.9187460766241071, bias : -0.1460205689063377, loss : 0.7946601314454398
epochs : 1, w1 : 0.9398401385990321, w2 : 0.8501471584489816, bias : -0.27204171877303784, loss : 0.7406483743374357
epochs : 2, w1 : 0.9206706229599357, w2 : 0.7936519337479107, bias : -0.37956883651373846, loss : 0.7013993799659705
epochs : 3, w1 : 0.9078393941627181, w2 : 0.7481703672678416, bias : -0.47065669931226084, loss : 0.6736461369021787
epochs : 4, w1 : 0.9004883151964336, w2 : 0.7123266329840607, bias : -0.5475768037802378, loss : 0.6543742239079764
epochs : 5, w1 : 0.8977622397819759, w2 : 0.6846746837538349, bias : -0.6125610961688313, loss : 0.6411041158940357
epochs : 6, w1 : 0.898872736530929, w2 : 0.6638408579081043, bias : -0.667647813466844, loss : 0.6319542107371117
epochs : 7, w1 : 0.903128085649941, w2 : 0.6485980813362056, bias : -0.714612956066349, loss : 0.6255741155703062
epochs : 8, w1 : 0.9099405444741201, w2 : 0.637891884131865, bias : -0.75

(np.float64(3.8243792453432754),
 np.float64(1.0424089248140604),
 np.float64(-2.4161388502721217))

In [41]:
coef, intercept

(array([[3.7773602],
        [1.1495365]], dtype=float32),
 array([-2.5030148], dtype=float32))